# 딥러닝의 개념과 작동 원리 2

## 학습방법

지도학습, 비지도학습, 자기지도학습, 강화학습

자기지도학습: 대규모 레이블 없는 데이터에 활용하는 학습 방법. BERT와 GPT가 있다.
> GPT를 지도학습으로 학습 시키는 것은 불가능하다. 학습하는 양이 수조개인데, 그것 하나하나 레이블(정답)을 줄 수 없다.  
> 그래서 GPT는 next word prediction으로, BERT는 mask를 채우는 방식으로 학습한다.
>
> 이 두 방식 모두 정답이 필요 없지만, 데이터 자체에서 지도 신호를 생성하기 때문에 마치 지도학습과 같은 학습이 이뤄지는 것이다


강화학습: 챗 지피티도 강화학습 한다! -> 환경과 상호작용하면서 보상을 최대화하는 학습법. 시행착오를 통해 최적의 행동 전략을 학습한다. 

지도학습을 생각해 보면,

익히 알다싶이 test, valid, train으로 데이터를 나눈다. 


이때 train 데이터 안에서 일어나는 일을 살펴보자

## 학습(training) 관련 용어

배치크기: 한 번에 모델에 넣는 즉, 한번에 학습하는 데이터 수(이때 나눈 조각조각을 미니배치라고 한다)  

이터레이션: 1 에폭을 끝내기 위한 배치 처리 횟수로, 매 이터레이션 마다 가중치(W)가 업데이트 된다(즉, 이터레이션은 하나의 배치를 사용하여 순전파 -> 손실 계산 -> 역전파 -> 가중치 업데이트를 마치는 한 번의 과정이다. 결국 이터레이션 크기 만큼 가중치를 업데이트 한다. )

에포크: '전체' 데이퍼 1회 순전파 & 역전파 완료.
> 1에폭이 끝나면 검증용 데이터로 검증을 한다. 안좋네? 그럼 다시 2에폭 시작. 정해진 에폭이 모두 끝나거나 허용치에 도달하면 test data로 최종 모델의 성능을 평가한다. 




좀 더 생각해 보면, 순전파와 역전파의 한 사이클이 1 이터레이션에서 일어난다. 
> iteration (반복): 하나의 배치를 사용하여 순전파 -> 손실 계산 -> 역전파 -> 가중치 업데이트를 마치는 한 번의 과정
>
> 

## 순전파 

입력층→ 은닉층→ 출력층 방향으로 한방향 연산
각층에서"선형변환(Wx+b) → 활성화 함수" 반복->  
최종 출력층에서 예측값(y-hat) 생성→ 이 예측값과
실제값의 차이를 손실함수로 계산.  
학습 초기에는 가중치가 무작위이므로 예측이 엉터리  
→ 역전파로 가중치를 조정해야함

$$Y = XW + B$$

이게 가장 간결한 공식이지만, 이거 하나하나 처리하지 않지!


$$\begin{bmatrix} y_{11} & y_{12} \\ y_{21} & y_{22} \\ y_{31} & y_{32} \\ y_{41} & y_{42} \end{bmatrix} = \begin{bmatrix} x_{11} & x_{12} & x_{13} \\ x_{21} & x_{22} & x_{23} \\ x_{31} & x_{32} & x_{33} \\ x_{41} & x_{42} & x_{43} \end{bmatrix} \times \begin{bmatrix} w_1 & w_4 \\ w_2 & w_5 \\ w_3 & w_6 \end{bmatrix} + \begin{bmatrix} b_1 & b_2 \end{bmatrix}$$




X: 입력행렬(4,3) 크기 즉, 4개의 샘플과 3개의 특성   
W: 가중치 행렬 (3,2) 크기 즉, 2개의 출력으로 연결  
B: 편향 벡터(각 출력 노드에 더해지는 값)  
Y: 출력 행렬 (4,2) 크기 즉, 4개 샘플에 대한 2개씩의 결과값(예측 또는 분류..)

*이때 B는 행렬의 크기가 다르지만, 컴퓨터가 알아서 모든 샘플에 행을 element wies 하게 더해준다. 즉, 브로드캐스팅!!

컴퓨터는 메모리를 아끼기 위해 실제로 $B$를 4번 복사해서 저장하지는 않지만, 논리적으로는 다음과 같이 행을 늘려서 계산한다. 


$$B = \begin{bmatrix} b_1 & b_2 \end{bmatrix} \quad \xrightarrow{\text{Broadcasting}} \quad \begin{bmatrix} b_1 & b_2 \\ b_1 & b_2 \\ b_1 & b_2 \\ b_1 & b_2 \end{bmatrix}$$



행렬 곱셈의 원리에 따라, 첫 번째 샘플의 첫 번째 출력($y_{11}$)이 나오는 과정은 다음과 같다.

$$y_{11} = (x_{11} \cdot w_1 + x_{12} \cdot w_2 + x_{13} \cdot w_3) + b_1$$



## 역전파 (가중치 업데이트) 


*역전파에서 중요한 것은 w를 고친다는 것이다. input x, 각 레이어의 output z, 활성화 함수 h 모두 아니라 w가 대상이다!*

dL/dW = dL/dy * dy/dz * dz/dW  
(연쇄법칙으로 각 가중치에 대한 기울기 계산)  
→ 이 기울기 방향으로 가중치를 조정

nabla(나블라)는 $\nabla$
-> 미분하라! 라는 연산자임. 근데, 벡터 대상으로 내리는 명령임!
> 나블라는 그 자체로 어떤 숫자를 의미하는 것이 아니라, "각 성분별로 편미분을 해서 벡터로 묶어라"라는 동작을 압축한 기호

 $\nabla L$이라는 표현은 **'모든 가중치에 대한 편미분 모음'**을 의미힌다. 즉, 존재하는 가중치가 w1 w2 w3라면. 


$$\nabla L = \left( \frac{\partial L}{\partial w_1}, \frac{\partial L}{\partial w_2}, \frac{\partial L}{\partial w_3} \right)$$


우리는 오차를 줄여야 하므로, 나블라가 가리키는 방향의 **반대($-\nabla L$)**로 가중치를 이동시키는 것이죠.

## 손실함수

MSE -> 회귀문제  
BCE -> 이진분류 문제(출력층에서 시그모이드를 사용하겠지)  
CCE -> 다중분류 문제(츨력층에서 소프트맥스를 사용하겠지)

